# einops.rearrange — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-rearrange`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `einops.rearrange` patterns that ramp from identity → axis swap → composition → decomposition → patching. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-rearrange`**, which bridges to the bank subtopic `Einops: Rearrange` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange"
DD_SUBTOPIC = "Einops: Rearrange"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## einops.rearrange — quick refresher

`rearrange(tensor, pattern, **axes_lengths)` does three things with one pattern:
1. **Reorder axes** — `'h w -> w h'` is a transpose.
2. **Compose axes** — `'h w c -> (h w) c'` flattens spatial dims into one.
3. **Decompose axes** — `'(b1 b2) c -> b1 b2 c'` splits one axis into two (requires `b1=` or `b2=`).

Identifiers on the right side must match identifiers on the left — every axis is named, every axis is accounted for. No transposing semantics beyond what the pattern says.

### Exercise 1 — identity rearrange

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall the pattern syntax for an identity rearrange of a 2-D tensor.
> Keywords: identity-pattern, axis-naming
> ```

**KCs targeted:** `rearrange-identity-pattern`

Implement `ex1_identity(x)` so it returns `x` rearranged by a pattern that leaves the layout unchanged. The input is a 2-D tensor of shape `(b, c)`.

Use `einops.rearrange` (not `.clone()` or `.contiguous()`) — the point is to write the pattern.

In [ ]:
def ex1_identity(x: Tensor) -> Tensor:
    """Rearrange `x` of shape (b, c) to the same shape (b, c)."""
    raise NotImplementedError()


def _test_ex1():
    x = t.arange(12).reshape(3, 4)
    y = ex1_identity(x)
    assert y.shape == x.shape, f'shape mismatch: {y.shape} vs {x.shape}'
    assert t.equal(y, x), 'values differ'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_identity(x: Tensor) -> Tensor:
    return rearrange(x, 'b c -> b c')
```
</details>

### Exercise 2 — axis swap (transpose)

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply the rearrange pattern syntax to perform a 2-D transpose.
> Keywords: transpose, axis-renaming, permutation
> ```

**KCs targeted:** `rearrange-axis-swap`

Implement `ex2_swap(x)` to swap the two axes of a 2-D tensor. Input shape `(rows, cols)`, output shape `(cols, rows)`.

This is equivalent to `x.T` — but write it as a rearrange pattern.

In [ ]:
def ex2_swap(x: Tensor) -> Tensor:
    """Rearrange `x` of shape (rows, cols) to shape (cols, rows)."""
    raise NotImplementedError()


def _test_ex2():
    x = t.arange(12).reshape(3, 4)
    y = ex2_swap(x)
    assert y.shape == (4, 3), f'expected (4,3), got {y.shape}'
    assert t.equal(y, x.T), 'values differ from x.T'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_swap(x: Tensor) -> Tensor:
    return rearrange(x, 'rows cols -> cols rows')
```
</details>

### Exercise 3 — image flatten (axis composition)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply axis composition `(c h w)` on the output side to flatten a 4-D tensor's trailing axes in row-major order.
> Keywords: composition, flatten, row-major
> ```

**KCs targeted:** `rearrange-axis-composition`

Implement `ex3_flatten(x)` to flatten a batch of CHW images into a batch of feature vectors.

Input shape: `(b, c, h, w)`. Output shape: `(b, c * h * w)`.

Use a **composed** axis on the right side: `(c h w)` collapses three named axes into one. Row-major order — channel varies slowest, width varies fastest.

In [ ]:
def ex3_flatten(x: Tensor) -> Tensor:
    """Rearrange (b, c, h, w) → (b, c*h*w) by composing the trailing 3 axes."""
    raise NotImplementedError()


def _test_ex3():
    x = t.arange(2 * 3 * 4 * 5).reshape(2, 3, 4, 5).float()
    y = ex3_flatten(x)
    assert y.shape == (2, 60), f'expected (2,60), got {y.shape}'
    # Row-major flatten should match torch.reshape exactly.
    assert t.equal(y, x.reshape(2, 60)), 'values differ from torch.reshape'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_flatten(x: Tensor) -> Tensor:
    return rearrange(x, 'b c h w -> b (c h w)')
```

**Why row-major?** `einops` composes axes in the order written. `(c h w)` means the stride pattern is `(c × h × w, h × w, w, 1)` — equivalent to `torch.reshape` on a contiguous tensor.
</details>

### Exercise 4 — batch unfold (axis decomposition)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply axis decomposition `(a b)` on the input side with a kwarg-bound size to split one axis into two.
> Keywords: decomposition, kwarg-binding, micro-batching
> ```

**KCs targeted:** `rearrange-axis-decomposition`

Implement `ex4_unfold(x, micro_batch_size)` to split the leading batch dimension into a `(num_micro, micro_batch_size)` pair.

Input shape: `(B, c)` where `B = num_micro × micro_batch_size`. Output shape: `(num_micro, micro_batch_size, c)`.

This is **decomposition** — one axis on the left becomes a parenthesized pair, with one side bound via a keyword argument. You'll need to pass `micro_batch_size` into `rearrange` as a named axis length.

In [ ]:
def ex4_unfold(x: Tensor, micro_batch_size: int) -> Tensor:
    """Rearrange (B, c) → (num_micro, micro_batch_size, c).

    Assumes B is divisible by micro_batch_size.
    """
    raise NotImplementedError()


def _test_ex4():
    x = t.arange(12 * 5).reshape(12, 5).float()
    y = ex4_unfold(x, micro_batch_size=4)
    assert y.shape == (3, 4, 5), f'expected (3,4,5), got {y.shape}'
    # The first micro-batch should be the first 4 rows of x.
    assert t.equal(y[0], x[:4]), 'micro-batch 0 does not match x[:4]'
    assert t.equal(y[1], x[4:8]), 'micro-batch 1 does not match x[4:8]'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_unfold(x: Tensor, micro_batch_size: int) -> Tensor:
    return rearrange(x, '(num_micro mb) c -> num_micro mb c', mb=micro_batch_size)
```

**Why pass `mb=`?** When you decompose an axis with `(a b)`, einops needs to know one of the two sizes — the other is inferred from the total length. The naming on left and right just needs to be consistent.
</details>

### Exercise 5 — patch grid (ViT-style patch embedding prep)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize axis decomposition, reordering, and composition into a single rearrange pattern that produces a ViT patch-embedding layout.
> Keywords: patchify, vit, integration, multi-kc
> ```

**KCs targeted:** `rearrange-axis-decomposition`, `rearrange-axis-composition`, `rearrange-combined-patterns`

Implement `ex5_patchify(x, patch_size)` to break a batch of images into a flat sequence of patches.

Input shape: `(b, c, H, W)` where `H` and `W` are both divisible by `patch_size`. Output shape: `(b, num_patches, patch_size * patch_size * c)` where `num_patches = (H // patch_size) * (W // patch_size)`.

This combines **decomposition** (split each spatial axis into `(h p1)` and `(w p2)`), **reordering** (move patch dims after grid dims), and **composition** (flatten grid into a sequence and pixels into a feature vector). It's the operation at the start of a Vision Transformer.

> ⚠️ **Integrative exercise.** This combines 3+ KCs in one pattern; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_patchify(x: Tensor, patch_size: int) -> Tensor:
    """Rearrange (b, c, H, W) → (b, num_patches, patch_size*patch_size*c).

    Patch order: row-major over the (h, w) patch grid.
    Pixel order inside a patch: row-major over (p1, p2), then channel.
    """
    raise NotImplementedError()


def _test_ex5():
    b, c, H, W, p = 2, 3, 8, 8, 4
    x = t.arange(b * c * H * W).reshape(b, c, H, W).float()
    y = ex5_patchify(x, patch_size=p)
    num_patches = (H // p) * (W // p)
    feat = p * p * c
    assert y.shape == (b, num_patches, feat), f'expected ({b},{num_patches},{feat}), got {y.shape}'

    # Round-trip: the first patch of the first image should be the top-left
    # p×p block across all c channels, flattened in (p1, p2, c) order.
    top_left = x[0, :, :p, :p]                      # (c, p, p)
    expected_patch0 = rearrange(top_left, 'c p1 p2 -> (p1 p2 c)')
    assert t.equal(y[0, 0], expected_patch0), 'first patch does not match top-left block'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_patchify(x: Tensor, patch_size: int) -> Tensor:
    return rearrange(
        x,
        'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
        p1=patch_size, p2=patch_size,
    )
```

**Reading the pattern.**
- `(h p1)` and `(w p2)` decompose H and W into (grid, patch) factor pairs. Pass `p1=` and `p2=` so the grid sizes can be inferred.
- `(h w)` on the right composes the grid into the sequence axis.
- `(p1 p2 c)` composes patch pixels and channels into the per-token feature vector. The order `(p1 p2 c)` matters — it determines the layout the downstream Linear layer sees. ViT papers usually write `(p1 p2 c)` so adjacent pixels are adjacent in the feature dim.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'   # 5/5 → felt easy
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()